搭建的所有神经网络都必须从nn.module继承

nn.Module 的 __init__ 方法会创建一些极其重要的内部数据结构，用来记录和管理你的网络层。__init__()在对子类进行赋值之前，必须先调用父类。

In [6]:
import torch.nn as nn
import torch.nn.functional as F
import torch


class Model(nn.Module):
    def __init__(self) -> None: #箭头none表示没有返回任何值
        super().__init__() #这里必须调用父类的构造函数
        self.conv1 = nn.Conv2d(1, 20, 5)
        self.conv2 = nn.Conv2d(20, 20, 5)

    def forward(self, x):#input->神经网格的计算(forward正向传播)->output,这里假设的输入是x
        x = F.relu(self.conv1(x))#对x先是卷积，然后进行一个非线性的处理
        return F.relu(self.conv2(x))#再进行一次卷积，再进行一次非线性才能得到一个输出

In [12]:
class ShenJing(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,input):
        output = input +1
        return output

shenjing =ShenJing()
x = torch.tensor(1.0) #创建一个0维张量，这是神经网络基本数据单位 
output = shenjing(x)
output

tensor(2.)

# 卷积 convolution

这是一个卷积核的示意图：

![卷积操作](./notes/convolution.png)



In [ ]:
输出尺寸 = (输入尺寸 - 卷积核大小 + 2 × padding) / 步长 + 1

In [25]:
input = torch.tensor([[1,2,0,3,1],#ypeError: tensor() takes 1 positional argument but 5 were given
                     [0,1,2,3,1], #tensor只能接受一个参数，所以要套两层【】
                     [1,2,1,0,0],
                     [5,2,3,1,1],
                     [2,1,0,1,1]])
#卷积核(weight)
kernel = torch.tensor([[1,2,1],
                      [0,1,0],
                      [2,1,0]])


# conv2d进行卷积计算,input和kernel必须要满足4个参数才行
print(input.shape)#打印出来发现只有高和宽，没有minibatch和in_chanels
print(kernel.shape)#打印出来发现只有高和宽，没有out_chanels和(in_chanels)/groups



# pytorch有提供一种尺寸变换
input = torch.reshape(input,(1,1,5,5))#batch大小，通道数，高，宽
kernel = torch.reshape(kernel,(1,1,3,3))
print(input.shape)
print(kernel.shape)
#如果out_chanel=2则会生成两个卷积核，对应两个output

output = F.conv2d(input,kernel,stride=1)#stride是步长的意思
print(output)
output2 = F.conv2d(input,kernel,stride=2)#stride是步长的意思
print(output2)
output3 = F.conv2d(input,kernel,stride=1,padding=1)#padding是在input周围加几圈，值通常为0
print(output3)


torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1, 5, 5])
torch.Size([1, 1, 3, 3])
tensor([[[[10, 12, 12],
          [18, 16, 16],
          [13,  9,  3]]]])
tensor([[[[10, 12],
          [13,  3]]]])
tensor([[[[ 1,  3,  4, 10,  8],
          [ 5, 10, 12, 12,  6],
          [ 7, 18, 16, 16,  8],
          [11, 13,  9,  3,  4],
          [14, 13,  9,  7,  4]]]])


# 2d卷积计算实例

In [ ]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader 
from torch.utils.tensorboard import SummaryWriter
from torch.nn import Conv2d

dataset = torchvision.datasets.CIFAR10("./data", train=False,transform=torchvision.transforms.ToTensor(),download = True)

dataloader =DataLoader(dataset,batch_size=64)#数据集，一批多大

class ShenJing(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = Conv2d(in_channels=3,out_channels=6,kernel_size=3,stride=1,padding=0)
        
    def forward(self,x):
        x = self.conv1(x)
        return x


shenjing = ShenJing()

writer = SummaryWriter("logs")
step =0
for data in dataloader:
    imgs,targets = data
    output = shenjing(imgs)#TypeError: __init__() takes 1 positional argument but 2 were given这里是调用实例对象而不是类
    writer.add_images("input",imgs,step)#input本身就是3通道
    output=torch.reshape(output,(-1,3,30,30))#-1：让 PyTorch 自动计算这个维度（64 × 6 ÷ 3 = 128）因为卷积之后变成6通道了
    writer.add_images("output",output,step)
    step+=1

writer.close()

# 池化层

这是一个最大池化操作的示意图：
与卷积核不同的是，默认步长是池化核的长度  
（pooling有汇集的意思）

![最大池化层](./notes/最大池化操作.png)

In [52]:
from torch.nn import MaxPool2d


input = torch.tensor([[1,2,0,3,1],
                     [0,1,2,3,1], 
                     [1,2,1,0,0],
                     [5,2,3,1,1],
                     [2,1,0,1,1]])
input = torch.reshape(input,(1,1,5,5))#batch大小，通道数，高，宽


class Pooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.maxpool1 =MaxPool2d(3,ceil_mode=True)
    def forward(self,input):
        output = self.maxpool1(input)
        return output

pool = Pooling()
output = pool(input)
print(output)







tensor([[[[2, 3],
          [5, 1]]]])


### 池化层的作用
* 池化就是在减少特征图尺寸的同时，把每个局部区域里最"拿手"的特征给提炼出来。

1. "收集特征" = 提炼/保留主要特征

2. "减少大小" = 降维/降低计算量

# 非线性激活

1. 线性变换 vs. 非线性变换
首先，想象一下，你只能用直线和平面来思考问题。

线性变换：就像y = 2x或者y = ax + b。无论你怎么组合（加、乘），结果永远是一条直线或一个平面。它只能解决简单的问题，比如“价格越高，销量越低”这种单调关系。

神经网络如果只有线性变换会怎样？
无论你堆叠多少层，最终它都等价于一个单层的线性模型。它的学习能力非常有限，甚至无法解决像“异或（XOR）”这样简单的逻辑问题。

非线性变换：就像y = x²或者y = sin(x)。它能让直线变弯，让平面变成曲面。

非线性激活的作用就是：为神经网络引入“弯曲”的能力。
正是这种“弯曲”的能力，让神经网络可以去拟合任何复杂的函数，去识别图像里的猫、理解句子里的情感、预测股票市场的波动。

一句话总结：没有非线性激活，神经网络就只是一个“高级线性回归模型”。

2. 激活函数：决定“神经元”是否被激活
“激活”这个词很形象。你可以把神经网络里的每个神经元想象成一个小灯泡。

激活函数就是这个灯泡的开关。它接收上一个神经元传来的信号（一个数值），然后决定：

这个信号够不够强，值不值得“点亮”这个神经元？
如果点亮了，应该以多强的亮度（输出一个什么数值）传递给下一个神经元？
它把这个“开关”规则用一种数学函数的形式固定下来。而非线性激活函数，就是这个“开关”的规则是非线性的。  


# 常用激活函数

### ReLU
* ReLU 就是把负数变成0，正数保持不变

In [61]:
import torch
from torch import nn
from torch.nn import ReLU
#ImportError: cannot import name 'ReLU' from 'torch' (/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/__init__.py)
#ReLU是在torch.nn下面而不是torch下面


#torch.tensor() 是一个构造函数，用来从零创建张量。而 ToTensor() 是一个数据转换操作，专门用来把 PIL 图片或 NumPy 数组 转换成张量。
input = torch.tensor([[1,-0.5],
                      [-1,3]
                     ])

input= torch.reshape(input,(-1,1,2,2))#-1表示自动算

class Shen(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu1 = ReLU()#初始化没有参数
    def forward(self,input):
        output= self.relu1(input)#inplace=False的时候需要把返回值赋值给一个变量（output），只有变量会被ReLU而input不会。默认是Flase
        return output


shen = Shen()
output = shen(input)
print(output)

tensor([[[[1., 0.],
          [0., 3.]]]])


# 小实战

![CIFAR10](./notes/cifar10.png)

In [77]:
from torch.nn import Flatten,Conv2d,MaxPool2d,Linear,Sequential


class Shen(nn.Module):
    def __init__(self):
        super().__init__()
        
        # #卷积层
        # self.conv1 = Conv2d(3,32,5,padding=2,stride=1)#(in_channels,out_channels,kernel，padding，stride)根据官网公式与前三项得到stride和padding

        # #池化层
        # self.maxpool1 = MaxPool2d(2)#kernel

        # #卷积层
        # self.conv2 = Conv2d(32,32,5,padding=2,stride=1)

        # #池化层
        # self.maxpool2 = MaxPool2d(2)#kernel

        # #卷积层
        # self.conv3 = Conv2d(32,64,5,padding=2,stride=1)
            
        # #池化层
        # self.maxpool3 = MaxPool2d(2)#kernel

        # #flatten过渡层，将多维张量"压平"成一维的层，通常用在卷积层到全连接层的过渡。
        # self.flatten =Flatten()

        # #线性层
        # self.linear1 = Linear(1024,64)#忘了参数多少可以把forward之后的删掉看输出的格式
        
        # #线性层
        # self.linear2 = Linear(64,10) #分10个类别
        
        
        #用Sequential管理所有层
        self.model1 = Sequential(
             Conv2d(3,32,5,padding=2,stride=1),
             MaxPool2d(2),
             Conv2d(32,32,5,padding=2,stride=1),
             MaxPool2d(2),
             Conv2d(32,64,5,padding=2,stride=1),
             MaxPool2d(2),
             Flatten(),
             Linear(1024,64),
             Linear(64,10)
        )
        
    def forward(self,x):
        # x=self.conv1(x)
        # x=self.maxpool1(x)
        # x=self.conv2(x)
        # x=self.maxpool2(x)
        # x=self.conv3(x)
        # x=self.maxpool3(x)
        # x=self.flatten(x)
        # x=self.linear1(x)
        # x=self.linear2(x)

        x=self.model1(x)
        return x
        

shen =Shen()
print(shen)

# #简单检验网络是否出错，有输出网络参数错了这里就会报错
# input =torch.ones(64,3,32,32)#(batch_size,channels,h,w)
# output = shen(input)
# print(output.shape)
writer =SummaryWriter("log_seq")
input =torch.ones(64,3,32,32)#(batch_size,channels,h,w)
writer.add_graph(shen,input)#(model,input)
writer.close()



Shen(
  (model1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1024, out_features=64, bias=True)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)


# 损失函数与反向传播

![lost](./notes/lost.png)

In [98]:
from torch.nn import L1Loss
import torch

inputs =torch.tensor([1,2,3],dtype=torch.float32)
targets = torch.tensor([1,2,5],dtype=torch.float32)
#使用损失函数的时候一定要注意他要的输入是什么样，输出是什么样的
inputs = torch.reshape(inputs,(1,1,1,3))
targets = torch.reshape(targets,(1,1,1,3))

loss = L1Loss()
result = loss(inputs,targets)

loss_mse = nn.MSELoss()
result_mse = loss_mse(inputs,targets)

#多分类问题
x = torch.tensor([0.1,0.2,0.3])
y = torch.tensor([1])
x = torch.reshape(x,(1,3))#batch_size=1,class=3
loss_cross = nn.CrossEntropyLoss()
result_cross = loss_cross(x,y)

# RuntimeError: mean(): could not infer output dtype. Input dtype must be either a floating point or complex dtype. Got: Long
print(result)#tensor(0.6667)  差距是2，总数是3，三分之二
print(result_mse)#tensor(1.3333) 平方相加再除以总数，1,333
print(result_cross)#tensor(1.1019)损失值 1.1019 表示模型对真实类别的预测概率不够高（只有33.22%）
                   #理想情况：如果模型对真实类别的预测概率接近1，损失会接近0如果预测错误（真实类别概率接近0），损失会趋向无穷大
                   #第一步：e^0.1 = 1.105171 ,e^0.2 = 1.221403  ,e^0.3 = 1.349859
                   #第二步，把三个相加，再除以各自值就是概率得到是第二类（1）的概率是33.22%。这只是各类别的概率而不是正确预测率

tensor(0.6667)
tensor(1.3333)
tensor(1.1019)


# CrossEntropyLoss损失函数内置的SoftMax激活函数
### 为什么要用 e^x？
1. 因为原始 logits [0.1, 0.2, 0.3]
    * 可能是负数
    * 让差距大的更大，差距小的更小
    * 软"不是指"让差距变小"，而是指"可微、连续、有概率解释"
2. 保证所有概率为正数（e^x > 0）
3. 保持大小关系：logits 越大，概率越大
### 指数运算后得到的损失值与预测正确率的关系
Loss = -ln(P) = 2.3  
ln(P) = -2.3   
P = e^(-2.3) = 0.1003  # 约 10%

# 损失函数在神经网络中的使用

In [110]:

dataset = torchvision.datasets.CIFAR10("data",train = False,transform = torchvision.transforms.ToTensor(),download =True)
dataloader =DataLoader(dataset,batch_size=1)
class Shen(nn.Module):
    def __init__(self):
        super().__init__()
        self.model1 = Sequential(
             Conv2d(3,32,5,padding=2,stride=1),
             MaxPool2d(2),
             Conv2d(32,32,5,padding=2,stride=1),
             MaxPool2d(2),
             Conv2d(32,64,5,padding=2,stride=1),
             MaxPool2d(2),
             Flatten(),
             Linear(1024,64),
             Linear(64,10)
        )
        
    def forward(self,x):
        x=self.model1(x)
        return x

loss = nn.CrossEntropyLoss()
shen = Shen()
for data in dataloader:
    imgs,targets=data
    outputs = shen(imgs)#通过神经网络得到输出
    result_loss = loss(outputs,targets)
    #反向传播
    result_loss.backward()#这样才能计算各个节点的参数包括grad
    # print(result_loss)

Files already downloaded and verified


# 优化器

In [ ]:
import torch
import torchvision
from torch import nn
from torch.nn import Flatten,Conv2d,MaxPool2d,Linear,Sequential
from torch.utils.data import DataLoader

dataset = torchvision.datasets.CIFAR10("data",train = False,transform = torchvision.transforms.ToTensor(),download =True)
dataloader =DataLoader(dataset,batch_size=1)


class Shen(nn.Module):
    def __init__(self):
        super().__init__()
        self.model1 = Sequential(
             Conv2d(3,32,5,padding=2,stride=1),
             MaxPool2d(2),
             Conv2d(32,32,5,padding=2,stride=1),
             MaxPool2d(2),
             Conv2d(32,64,5,padding=2,stride=1),
             MaxPool2d(2),
             Flatten(),
             Linear(1024,64),
             Linear(64,10)
        )
        
    def forward(self,x):
        x=self.model1(x)
        return x

loss = nn.CrossEntropyLoss()
shen = Shen()
#先定义一个优化器
optim = torch.optim.SGD(shen.parameters(),lr = 0.01)
for epoch in range(5):
    running_loss =0.0
    for data in dataloader:
        imgs,targets=data
        outputs = shen(imgs)
        result_loss = loss(outputs,targets)
        #第一步，把网络中所有元素梯度调0
        optim.zero_grad()
        #第二步，需要反向传播计算出grad
        result_loss.backward()
        #第三步，调用优化器
        optim.step()
        #print(result_loss)变化不是很大因为一层循环只能进行一轮的学习
        running_loss = running_loss+result_loss
    print(running_loss)


Files already downloaded and verified
tensor(84243.4219, grad_fn=<AddBackward0>)
tensor(88552.4922, grad_fn=<AddBackward0>)


# 梯度下降算法

![梯度下降算法](./notes/梯度下降算法.png)